# Predicting Annual Revenue of Rural Water Infrastructure in Africa

## 1. Introduction

### Mission Statement
To leverage software engineering and data-driven technologies to promote transparency, accountability, and efficient public service delivery, contributing to efforts to combat corruption and strengthen good governance in Uganda and across Africa.

### Problem Statement
Rural water service providers often face challenges in estimating the revenue generated by their infrastructure, making financial planning and resource allocation difficult [1], [2]. Without reliable estimates, it becomes challenging to plan for infrastructure maintenance, operational costs, and sustainable water service delivery [3]. Using machine learning to estimate annual water-service revenue based on infrastructure characteristics, population served, and revenue collection methods could support informed decision making and financial accountability.

### Project Objectives
- Exploring and preprocessing rural water infrastructure data to identify factors influencing annual revenue.
- Developing and comparing four regression models: SGDRegressor, Linear Regression, Decision Tree, and Random Forest.
- Evaluating model performance using Mean Squared Error (MSE) and selecting the best performing model.
- Deploying the selected model through a public FastAPI service on Render, supporting predictions and retraining.
- Developing a Flutter mobile application that allows users to input infrastructure characteristics and obtain estimated annual revenue.

### Intended Users
- Water service operators
- Public infrastructure planners

### Dataset Overview
- Source: [Dryad](https://datadryad.org/dataset/doi:10.5061/dryad.7pvmcvf7r)
- Dataset Name: Revenue and investment data for rural drinking water infrastructure in Africa 2021-2024
- Author: Duncan McNicholl, University of Oxford
- Geographical coverage: 11 African countries
- Period: 2021-2024
- Observations: 11,957
- Columns: 44
- Target variable: wp_revenue_USD

### References
[1] D. McNicholl, E. Aydinsoy, and R. Hope, “Africa’s water revenue paradox,” Research Square, preprint, Mar. 2026, doi: 10.21203/rs.3.rs-8878999/v1.

[2] A. M. Armstrong, “Revenue patterns of piped water services in rural Africa,” D.Phil. dissertation, Univ. of Oxford, Oxford, U.K., 2022, doi: 10.5287/ora-2r7vvmmpb.

[3] D. S. G. Agueh, “Financing water and sanitation in Africa: Resilience, reuse, ecosystems, decentralised services and water diplomacy,” RFLD, Oct. 9, 2026. [Online]. Available: https://rflgd.org/financing-water-sanitation-africa-gloria-agueh/.


## 2. Data Loading and Initial Inspection

## 2.1 Importing libraries

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2.2 Loading dataset

In [5]:
df = pd.read_csv('data/analysis_dataset.csv')

## 2.3 Initial dataset inspection

In [15]:
print(df.shape) # Confirming dataset dimensions

df.head() # Previewing first 5 observations

df.info() # Inspecting data types and non-null counts

df.describe() # Examining numerical distributions

(11957, 44)
<class 'pandas.DataFrame'>
RangeIndex: 11957 entries, 0 to 11956
Data columns (total 44 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   year                           11957 non-null  int64  
 1   lat                            11957 non-null  float64
 2   lon                            11957 non-null  float64
 3   num_taps                       11957 non-null  int64  
 4   num_hh                         11957 non-null  int64  
 5   wp_pop_last                    11957 non-null  int64  
 6   num_kiosks                     8071 non-null   float64
 7   num_scheme_tanks               3561 non-null   float64
 8   wp_revenue_model               11525 non-null  str    
 9   volume_method                  8071 non-null   str    
 10  country_name                   11957 non-null  int64  
 11  wp_m3                          11957 non-null  float64
 12  wp_revenue_USD                 11957 non-null

,year,lat,lon,num_taps,num_hh,wp_pop_last,num_kiosks,num_scheme_tanks,country_name,wp_m3,...,pop_served_last_within_2000m,pop_served_last_within_5000m,pop_served_last_within_10000m,pop_served_last_within_20000m,Population_Within_500m,Population_Within_1000m,Population_Within_2000m,Population_Within_5000m,Population_Within_10000m,Population_Within_20000m
count,11957.000000,11957.0,11957.0,11957.000000,11957.000000,11957.000000,8071.000000,3561.000000,11957.000000,11957.000000,...,11891.000000,11891.000000,11891.000000,11891.000000,11898.000000,11898.000000,11898.000000,11898.000000,1.189800e+04,1.189800e+04
mean,2022.837585,0.0,0.0,0.556410,3.582086,788.350924,0.065549,0.019938,4.405620,992.753307,...,7750.337987,20905.691111,44119.843916,79195.244302,937.563018,2226.958525,5108.264345,18034.726023,5.772356e+04,1.972405e+05
std,0.879613,0.0,0.0,2.285919,32.862670,1645.037532,1.234582,0.156851,3.419542,3051.982047,...,14058.411701,36377.288084,68471.814265,87777.140270,1470.087582,3714.054648,9035.857821,36338.479652,1.025141e+05,2.927969e+05
min,2021.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,2022.000000,0.0,0.0,0.000000,0.000000,197.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,1878.000000,8020.000000,23804.000000,192.680252,453.261508,1004.266962,3304.601272,1.104608e+04,3.589454e+04
50%,2023.000000,0.0,0.0,0.000000,0.000000,282.000000,0.000000,0.000000,3.000000,0.000000,...,2000.000000,7880.000000,22560.000000,53356.000000,451.838684,998.909635,2575.468561,10689.994140,3.172623e+04,7.608601e+04
75%,2024.000000,0.0,0.0,0.000000,0.000000,672.000000,0.000000,0.000000,9.000000,982.800000,...,7784.000000,20738.000000,47560.000000,98786.000000,1080.684444,2292.966191,5547.358155,21419.958132,6.562891e+04,2.458513e+05
max,2024.000000,0.0,0.0,44.000000,888.000000,23708.000000,44.000000,3.000000,10.000000,76673.120000,...,141960.000000,252457.000000,402122.000000,421365.000000,25361.761850,74163.978840,246267.334600,815807.769200,1.828080e+06,3.338384e+06


## 2.4 Initial observations
- The dataset contains 11,957 observations and 44 columns, covering 2021-2024.
- It includes numerical and categorical features representing rural water infrastructure.
- The target variable is wp_revenue_USD, representing annual revenue in USD.
- Some columns contain missing values that will require preprocessing.
- The latitude and longitude columns are anonymized for confidentiality.
- The twelve monthly variables contain no numerical variation.